# 2. Analysis: 시계열 분석 + 다수준 로지스틱 회귀

| 항목 | 내용 |
|------|------|
| **입력** | `output/pre_output/analytic_sample.csv` (1_preprocess 출력) |
| **출력** | `output/analysis_output/` (테이블 CSV, 그림 PNG) |
| **분석** | (A) 시간 추세 (B) Survey-weighted GLM (C) Sensitivity |

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — 환경 설정 + 데이터 로드
# ══════════════════════════════════════════════════════════════
# Environment setup (adjust paths as needed)

import pandas as pd, numpy as np, os, warnings
from datetime import datetime
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
warnings.filterwarnings('ignore')

BASE    = os.path.abspath(os.path.join('..'))  # repository root
PRE_DIR = os.path.join(BASE, 'output', 'pre_output')

# 실행할 때마다 타임스탬프 폴더 생성 → 이전 결과와 비교 가능
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
OUT_BASE = os.path.join(BASE, 'output', 'analysis_output')
OUT_DIR  = os.path.join(OUT_BASE, f'run_{timestamp}')
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PRE_DIR, 'analytic_sample.csv'), encoding='utf-8-sig')
YEARS = {2:2010, 3:2011, 4:2014, 5:2017, 6:2020, 7:2023}

# 기존 실행 목록 출력
prev_runs = sorted([d for d in os.listdir(OUT_BASE)
                    if os.path.isdir(os.path.join(OUT_BASE, d)) and d.startswith('run_')])
print(f'로드 완료: N={len(df):,}')
print(f'출력 폴더: {OUT_DIR}')
if prev_runs:
    print(f'이전 실행: {len(prev_runs)}개 ({prev_runs[0]} ~ {prev_runs[-1]})')
else:
    print('이전 실행: 없음 (첫 실행)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1b — DAG (Directed Acyclic Graph)
# ══════════════════════════════════════════════════════════════
# 공변량 선택 근거를 시각화. 리뷰어가 "왜 이 변수를 보정했는가?"
# 질문 시 DAG를 제시하여 최소충분조정세트(minimal sufficient adjustment set)
# 근거를 명시. OEM/SJWEH 2023년부터 권장 추세.
#
# 참고: Li et al. 2023, Medicine; Textor et al. 2016, Int J Epidemiol
# ══════════════════════════════════════════════════════════════

import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 8.5)
ax.axis('off')
ax.set_title('Figure 1. Directed Acyclic Graph (DAG) for Covariate Selection',
             fontsize=14, fontweight='bold', pad=20)

# ── 노드 정의 (x, y, label, color) ──
nodes = {
    # Exposure (왼쪽)
    'PSY':    (1.5, 6.0, 'Psychosocial\nExposures\n(JDC-S)', '#3498db'),
    # Outcome (오른쪽)
    'MSD':    (8.5, 6.0, 'Musculoskeletal\nDisorders\n(MSD)', '#e74c3c'),
    # Confounders (중간/위)
    'AGE':    (5.0, 8.0, 'Age, Sex', '#7f8c8d'),
    'SES':    (3.0, 8.0, 'Education', '#7f8c8d'),
    'EMP':    (7.0, 8.0, 'Employment\ntype', '#7f8c8d'),
    'IND':    (5.0, 4.0, 'Industry', '#f39c12'),
    # Mediator/Confounder
    'ERG':    (5.0, 6.0, 'Ergonomic\nExposures', '#27ae60'),
    # Time
    'WAVE':   (1.5, 4.0, 'Wave\n(Time)', '#9b59b6'),
    # Work hours
    'HRS':    (8.5, 4.0, 'Long\nwork hours', '#7f8c8d'),
}

# 노드 그리기
for key, (x, y, label, color) in nodes.items():
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.15, edgecolor=color, linewidth=2)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold',
            bbox=bbox, color=color)

# ── 화살표 정의 (from, to, style) ──
arrows = [
    # Causal path of interest
    ('PSY', 'MSD', {'color': '#3498db', 'linewidth': 3, 'linestyle': '-'}),
    # Confounders → Exposure
    ('AGE', 'PSY', {'color': '#7f8c8d', 'linewidth': 1.5, 'linestyle': '--'}),
    ('SES', 'PSY', {'color': '#7f8c8d', 'linewidth': 1.5, 'linestyle': '--'}),
    ('IND', 'PSY', {'color': '#f39c12', 'linewidth': 2, 'linestyle': '-'}),
    ('WAVE', 'PSY', {'color': '#9b59b6', 'linewidth': 1.5, 'linestyle': '--'}),
    # Confounders → Outcome
    ('AGE', 'MSD', {'color': '#7f8c8d', 'linewidth': 1.5, 'linestyle': '--'}),
    ('EMP', 'MSD', {'color': '#7f8c8d', 'linewidth': 1.5, 'linestyle': '--'}),
    ('IND', 'MSD', {'color': '#f39c12', 'linewidth': 2, 'linestyle': '-'}),
    ('HRS', 'MSD', {'color': '#7f8c8d', 'linewidth': 1.5, 'linestyle': '--'}),
    ('WAVE', 'MSD', {'color': '#9b59b6', 'linewidth': 1.5, 'linestyle': '--'}),
    # Ergonomic: confounder on PSY→MSD path + direct effect
    ('ERG', 'MSD', {'color': '#27ae60', 'linewidth': 2, 'linestyle': '-'}),
    ('IND', 'ERG', {'color': '#f39c12', 'linewidth': 1.5, 'linestyle': '-'}),
    ('ERG', 'PSY', {'color': '#27ae60', 'linewidth': 1.5, 'linestyle': '-'}),
    # Industry → Employment
    ('IND', 'EMP', {'color': '#f39c12', 'linewidth': 1, 'linestyle': '--'}),
    ('EMP', 'PSY', {'color': '#7f8c8d', 'linewidth': 1.5, 'linestyle': '--'}),
    ('HRS', 'PSY', {'color': '#7f8c8d', 'linewidth': 1, 'linestyle': '--'}),
]

for frm, to, style in arrows:
    x1, y1 = nodes[frm][0], nodes[frm][1]
    x2, y2 = nodes[to][0], nodes[to][1]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=style['linewidth'],
                                color=style['color'], linestyle=style['linestyle'],
                                shrinkA=25, shrinkB=25, connectionstyle='arc3,rad=0.1'))

# ── 범례 ──
legend_items = [
    mpatches.Patch(color='#3498db', alpha=0.3, label='Exposure (JDC-S)'),
    mpatches.Patch(color='#e74c3c', alpha=0.3, label='Outcome (MSD)'),
    mpatches.Patch(color='#27ae60', alpha=0.3, label='Ergonomic (confounder)'),
    mpatches.Patch(color='#f39c12', alpha=0.3, label='Industry (key confounder)'),
    mpatches.Patch(color='#7f8c8d', alpha=0.3, label='Demographics/covariates'),
    mpatches.Patch(color='#9b59b6', alpha=0.3, label='Time (secular trend)'),
]
ax.legend(handles=legend_items, loc='lower left', fontsize=9, framealpha=0.9,
          title='Minimal Sufficient Adjustment Set', title_fontsize=10)

# ── 주석 ──
ax.text(5.0, -0.2,
    'Adjustment set: Age, Sex, Education, Employment type, Industry,\n'
    'Ergonomic exposures (4), Long work hours, Wave (time trend)\n'
    'Note: Industry is the key confounder explaining the "autonomy paradox"',
    ha='center', va='top', fontsize=9, style='italic', color='#555')

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_dag.png'), dpi=300, bbox_inches='tight')
plt.show()
print('저장: fig_dag.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — (A) 시간 추세 분석: 웨이브별 유병률
# ══════════════════════════════════════════════════════════════
# MSD 결과변수와 심리사회적 위험요인의 시계열 유병률 산출
# 가중치 적용하여 모집단 대표성 반영

outcomes  = ['msd_any','msd_back_b','msd_upper_b','msd_lower_b']
exposures = ['psy_demand_hi','lo_auto','lo_sup','job_strain','iso_strain']
all_vars  = outcomes + exposures

trend = []
for w in sorted(df['wave'].unique()):
    sub = df[df['wave']==w]
    row = {'wave': int(w), 'year': YEARS[int(w)], 'N': len(sub)}
    for v in all_vars:
        col = sub[v].dropna()
        wts = sub.loc[col.index, 'swt']
        row[f'{v}_prev'] = np.average(col, weights=wts) * 100  # 가중 유병률 %
    trend.append(row)

df_trend = pd.DataFrame(trend)
df_trend.to_csv(os.path.join(OUT_DIR, 'table_trend.csv'), index=False)
print('Table: 웨이브별 가중 유병률 (%)')
print(df_trend[['wave','year','N'] + [f'{v}_prev' for v in outcomes]].round(1).to_string(index=False))

# ── 추세 그림 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSD 추세
ax = axes[0]
for v, label in zip(outcomes, ['Any MSD','Back','Upper limb','Lower limb']):
    ax.plot(df_trend['year'], df_trend[f'{v}_prev'], 'o-', label=label)
ax.set_xlabel('Year'); ax.set_ylabel('Weighted prevalence (%)')
ax.set_title('(A) MSD Prevalence Trends, KWCS 2010-2023')
ax.legend(); ax.grid(alpha=0.3)

# 심리사회적 추세
ax = axes[1]
for v, label in zip(exposures, ['Hi demand','Lo autonomy','Lo support','Job strain','Iso-strain']):
    ax.plot(df_trend['year'], df_trend[f'{v}_prev'], 's--', label=label)
ax.set_xlabel('Year'); ax.set_ylabel('Weighted prevalence (%)')
ax.set_title('(B) Psychosocial Risk Factor Trends, KWCS 2010-2023')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_trends.png'), dpi=300, bbox_inches='tight')
plt.show()
print('저장: fig_trends.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — (B) Modified Poisson Regression: Prevalence Ratio
# ══════════════════════════════════════════════════════════════
#
# [D14] Modified Poisson regression (Zou, 2004)
#   - 횡단면 연구에서 유병률 >10% 시 OR은 과대추정 (Zhang & Yu, 1998)
#   - PR = Prevalence Ratio (family=Poisson, link=log, robust SE)
#   - Lancet Regional Health, OEM, SJWEH 모두 PR 권장
#
# [D15] 두 모델군:
#   Model A: 연속형 JDC-S (psy_demand, autonomy, support)
#   Model B: 복합 지표 (job_strain, iso_strain)
#
# [D16] 공변량:
#   인구통계: age_c, age_c2, female, edu_hi, long_hrs
#   인체공학: erg_posture_hi, erg_heavy_hi, erg_stand_hi, erg_repeat_hi
#   직업구조: C(emp_type), C(industry)
#   시간추세: wave_c
#
# [D17] 척도 방향:
#   psy_demand: 1=항상(고요구)~6=거의없음 → 방향 반전 보고
#   autonomy: 1=자율있음~2=없음 → 1점 증가 = 자율 감소
#   support: 1=항상~5=없음 → 1점 증가 = 지지 감소
# ══════════════════════════════════════════════════════════════

dfa = df.dropna(subset=['autonomy','support','lo_auto','lo_sup',
                         'edu_hi','poor_wlb']).copy()
print(f'분석 표본 (complete psychosocial): N={len(dfa):,}')
print(f'Any MSD 유병률: {dfa["msd_any"].mean()*100:.1f}% → OR 과대추정 → PR 사용')

covars = 'age_c + age_c2 + female + edu_hi + long_hrs'
erg_covars = 'erg_posture_hi + erg_heavy_hi + erg_stand_hi + erg_repeat_hi'
full_controls = f'{covars} + {erg_covars} + wave_c + C(emp_type) + C(industry)'

outcomes_dict = {
    'msd_any':    'Any MSD',
    'msd_back_b': 'Back pain',
    'msd_upper_b':'Upper limb',
    'msd_lower_b':'Lower limb',
}

results = []

for outcome, olabel in outcomes_dict.items():
    prev = dfa[outcome].mean()*100

    # ── Model A: Continuous JDC-S (Modified Poisson) ──
    fa = f'{outcome} ~ psy_demand + autonomy + support + {full_controls}'
    ma = smf.glm(fa, data=dfa, family=sm.families.Poisson(link=sm.families.links.Log()),
                  freq_weights=dfa['swt']).fit(cov_type='HC1')  # Robust SE

    for var, label, direction in [
        ('psy_demand', 'Psy demand (per unit increase)', -1),
        ('autonomy',   'Low autonomy (per unit increase)', 1),
        ('support',    'Low support (per unit increase)', 1)]:
        PR = np.exp(ma.params[var] * direction)
        ci_raw = ma.conf_int().loc[var]
        if direction == -1:
            ci = np.exp(np.array([-ci_raw.iloc[1], -ci_raw.iloc[0]]))
        else:
            ci = np.exp(ci_raw)
        results.append({
            'Outcome': olabel, 'Model': 'A (Continuous)',
            'Exposure': label, 'Prevalence': round(prev,1),
            'PR': round(PR, 2), 'CI_lo': round(ci[0], 2), 'CI_hi': round(ci[1], 2),
            'p': round(ma.pvalues[var], 4),
        })

    # ── Model B: Composite (Modified Poisson) ──
    fb = f'{outcome} ~ job_strain + iso_strain + {full_controls}'
    mb = smf.glm(fb, data=dfa, family=sm.families.Poisson(link=sm.families.links.Log()),
                  freq_weights=dfa['swt']).fit(cov_type='HC1')

    for var, label in [('job_strain','Job strain'), ('iso_strain','Iso-strain')]:
        PR = np.exp(mb.params[var])
        ci = np.exp(mb.conf_int().loc[var])
        results.append({
            'Outcome': olabel, 'Model': 'B (Composite)',
            'Exposure': label, 'Prevalence': round(prev,1),
            'PR': round(PR, 2), 'CI_lo': round(ci.iloc[0], 2), 'CI_hi': round(ci.iloc[1], 2),
            'p': round(mb.pvalues[var], 4),
        })

df_results = pd.DataFrame(results)
df_results['PR_CI'] = df_results.apply(
    lambda r: f"{r['PR']:.2f} ({r['CI_lo']:.2f}-{r['CI_hi']:.2f})", axis=1)
df_results.to_csv(os.path.join(OUT_DIR, 'table_regression.csv'), index=False)

print('\nTable: Modified Poisson Regression — Prevalence Ratio (95% CI)')
print('Method: Poisson(log link) + Robust SE (Zou, 2004)')
print('Controls: age, sex, edu, long_hrs, erg(4), emp_type, industry, wave')
print(df_results[['Outcome','Model','Exposure','PR_CI','p']].to_string(index=False))
print(f'\n저장: table_regression.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4 — Forest Plot (PR, Modified Poisson)
# ══════════════════════════════════════════════════════════════

colors = {'Any MSD':'#1a1a2e', 'Back pain':'#c0392b',
          'Upper limb':'#2980b9', 'Lower limb':'#27ae60'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Model A: Continuous (per-unit PR) ──
ax = axes[0]
sub_a = df_results[df_results['Model']=='A (Continuous)'].copy()
sub_a = sub_a.sort_values(['Outcome','Exposure'], ascending=[True, True]).reset_index(drop=True)

y_pos, y_labels, y, prev_outcome = [], [], 0, None
for _, row in sub_a.iterrows():
    if prev_outcome and row['Outcome'] != prev_outcome: y += 0.5
    y_pos.append(y)
    y_labels.append(f"{row['Outcome']}  |  {row['Exposure'].replace(' (per unit increase)','')}")
    prev_outcome = row['Outcome']; y += 1

for i, (_, row) in enumerate(sub_a.iterrows()):
    c = colors[row['Outcome']]
    marker = 'D' if 'demand' in row['Exposure'].lower() else ('s' if 'auto' in row['Exposure'].lower() else 'o')
    alpha = 1.0 if row['p'] < 0.05 else 0.4
    ax.plot(row['PR'], y_pos[i], marker, color=c, markersize=9, alpha=alpha, zorder=3)
    ax.plot([row['CI_lo'], row['CI_hi']], [y_pos[i], y_pos[i]], '-', color=c, linewidth=2.5, alpha=alpha, zorder=2)
    ax.annotate(f"{row['PR']:.2f}", (row['CI_hi']+0.005, y_pos[i]), fontsize=8, va='center', color=c, alpha=alpha)

ax.axvline(x=1, color='#7f8c8d', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.set_yticks(y_pos); ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel('Prevalence Ratio per unit increase (95% CI)', fontsize=10)
ax.set_title('Model A: Continuous JDC-S Exposures', fontsize=12, fontweight='bold')
ax.set_xlim(0.65, 1.12); ax.grid(axis='x', alpha=0.2); ax.invert_yaxis()
ax.plot([], [], 'ko', markersize=8, alpha=1.0, label='p < 0.05')
ax.plot([], [], 'ko', markersize=8, alpha=0.4, label='p >= 0.05')
ax.legend(loc='lower left', fontsize=8, framealpha=0.8)

# ── Model B: Composite (binary PR) ──
ax = axes[1]
sub_b = df_results[df_results['Model']=='B (Composite)'].copy()
sub_b = sub_b.sort_values(['Outcome','Exposure'], ascending=[True, False]).reset_index(drop=True)

y_pos, y_labels, y, prev_outcome = [], [], 0, None
for _, row in sub_b.iterrows():
    if prev_outcome and row['Outcome'] != prev_outcome: y += 0.5
    y_pos.append(y)
    y_labels.append(f"{row['Outcome']}  |  {row['Exposure']}")
    prev_outcome = row['Outcome']; y += 1

for i, (_, row) in enumerate(sub_b.iterrows()):
    c = colors[row['Outcome']]
    marker = 's' if 'Job' in row['Exposure'] else 'D'
    alpha = 1.0 if row['p'] < 0.05 else 0.4
    ax.plot(row['PR'], y_pos[i], marker, color=c, markersize=9, alpha=alpha, zorder=3)
    ax.plot([row['CI_lo'], row['CI_hi']], [y_pos[i], y_pos[i]], '-', color=c, linewidth=2.5, alpha=alpha, zorder=2)
    ax.annotate(f"{row['PR']:.2f}", (row['CI_hi']+0.01, y_pos[i]), fontsize=8, va='center', color=c, alpha=alpha)

ax.axvline(x=1, color='#7f8c8d', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.set_yticks(y_pos); ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel('Prevalence Ratio (95% CI)', fontsize=10)
ax.set_title('Model B: Composite Indicators', fontsize=12, fontweight='bold')
ax.set_xlim(0.55, 1.45); ax.grid(axis='x', alpha=0.2); ax.invert_yaxis()
ax.plot([], [], 'ko', markersize=8, alpha=1.0, label='p < 0.05')
ax.plot([], [], 'ko', markersize=8, alpha=0.4, label='p >= 0.05')
ax.legend(loc='lower right', fontsize=8, framealpha=0.8)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_forest.png'), dpi=300, bbox_inches='tight')
plt.show()
print('저장: fig_forest.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5 — (C) Sensitivity Analysis (5개) + E-value
# ══════════════════════════════════════════════════════════════
#
# S1. Without industry/emp_type (교란 보정 전후 비교)
# S2. Unweighted (가중치 영향)
# S3. +erg_lift_ppl (5개 erg, reduced sample — C(industry) 제외, 수렴 문제)
# S4. 성별 층화 (Male / Female)
# S5. W5(2017) 제외 (설문 방법 변경 가능성)
# E-value: 미측정 교란 정량 평가
# ══════════════════════════════════════════════════════════════

sens_results = []

def fit_poisson(formula, data, weighted=True):
    kw = dict(family=sm.families.Poisson(link=sm.families.links.Log()))
    if weighted: kw['freq_weights'] = data['swt']
    return smf.glm(formula, data=data, **kw).fit(cov_type='HC1')

def extract_pr(mod, var, direction=1):
    PR = np.exp(mod.params[var] * direction)
    ci_raw = mod.conf_int().loc[var]
    if direction == -1:
        ci = np.exp(np.array([-ci_raw.iloc[1], -ci_raw.iloc[0]]))
    else:
        ci = np.exp(ci_raw)
    return round(PR,2), round(ci[0],2), round(ci[1],2)

vars_info = [('psy_demand','Psy demand',-1), ('autonomy','Autonomy',1), ('support','Support',1)]
base_formula = 'msd_any ~ psy_demand + autonomy + support'

# S1: Without industry/emp_type
f_s1 = f'{base_formula} + {covars} + {erg_covars} + wave_c'
ms1 = fit_poisson(f_s1, dfa)
for var,label,d in vars_info:
    PR,lo,hi = extract_pr(ms1, var, d)
    sens_results.append({'Check':'S1: No ind/emp','Exposure':label,'PR':PR,'CI_lo':lo,'CI_hi':hi})

# S2: Unweighted
f_s2 = f'{base_formula} + {full_controls}'
ms2 = fit_poisson(f_s2, dfa, weighted=False)
for var,label,d in vars_info:
    PR,lo,hi = extract_pr(ms2, var, d)
    sens_results.append({'Check':'S2: Unweighted','Exposure':label,'PR':PR,'CI_lo':lo,'CI_hi':hi})

# S3: +erg_lift_ppl (reduced sample)
# C(industry) 제외: reduced sample에서 산업별 셀 크기 부족 → 수렴 문제
dfa_s3 = dfa.dropna(subset=['erg_lift_ppl_hi']).copy()
erg5 = 'erg_posture_hi + erg_lift_ppl_hi + erg_heavy_hi + erg_stand_hi + erg_repeat_hi'
f_s3 = f'{base_formula} + {covars} + {erg5} + wave_c + C(emp_type)'
ms3 = fit_poisson(f_s3, dfa_s3)
for var,label,d in vars_info:
    PR,lo,hi = extract_pr(ms3, var, d)
    sens_results.append({'Check':f'S3: +erg_lift (N={len(dfa_s3):,})','Exposure':label,'PR':PR,'CI_lo':lo,'CI_hi':hi})

# S4: 성별 층화
for sex_label, sex_val in [('S4a: Male only', 0), ('S4b: Female only', 1)]:
    dfa_sex = dfa[dfa['female']==sex_val].copy()
    covars_nosex = 'age_c + age_c2 + edu_hi + long_hrs'
    f_s4 = f'{base_formula} + {covars_nosex} + {erg_covars} + wave_c + C(emp_type) + C(industry)'
    ms4 = fit_poisson(f_s4, dfa_sex)
    for var,label,d in vars_info:
        PR,lo,hi = extract_pr(ms4, var, d)
        sens_results.append({'Check':f'{sex_label} (N={len(dfa_sex):,})','Exposure':label,'PR':PR,'CI_lo':lo,'CI_hi':hi})

# S5: W5(2017) 제외
dfa_s5 = dfa[dfa['wave']!=5].copy()
f_s5 = f'{base_formula} + {full_controls}'
ms5 = fit_poisson(f_s5, dfa_s5)
for var,label,d in vars_info:
    PR,lo,hi = extract_pr(ms5, var, d)
    sens_results.append({'Check':f'S5: Excl W5 (N={len(dfa_s5):,})','Exposure':label,'PR':PR,'CI_lo':lo,'CI_hi':hi})

df_sens = pd.DataFrame(sens_results)
df_sens['PR_CI'] = df_sens.apply(lambda r: f"{r['PR']:.2f} ({r['CI_lo']:.2f}-{r['CI_hi']:.2f})", axis=1)
df_sens.to_csv(os.path.join(OUT_DIR, 'table_sensitivity.csv'), index=False)

print('Table: Sensitivity Analysis — Prevalence Ratio (Any MSD)')
print(df_sens[['Check','Exposure','PR_CI']].to_string(index=False))

# ── E-value (VanderWeele & Ding, 2017) ──
def e_value(pr):
    if pr >= 1:
        return round(pr + np.sqrt(pr * (pr - 1)), 2)
    else:
        pr_inv = 1 / pr
        return round(pr_inv + np.sqrt(pr_inv * (pr_inv - 1)), 2)

print('\n' + '='*50)
print('E-values (VanderWeele & Ding, 2017)')
print('='*50)
print('미측정 교란이 관찰된 연관을 완전히 설명하려면 필요한 최소 RR')
primary = df_results[(df_results['Model']=='A (Continuous)') & (df_results['Outcome']=='Any MSD')]
for _, row in primary.iterrows():
    ev = e_value(row['PR'])
    print(f"  {row['Exposure']:40s} PR={row['PR']:.2f}  E-value={ev}")

print(f'\n저장: table_sensitivity.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — W5(2017) MSD 급감 원인 분석
# ══════════════════════════════════════════════════════════════
# Trend plot에서 W5(2017)의 Any MSD가 49%→31%로 급감.
# 리뷰어가 반드시 물을 포인트. 가능한 원인을 체계적으로 검토.
# ══════════════════════════════════════════════════════════════

print('='*70)
print('W5 (2017) MSD 급감 원인 분석')
print('='*70)

YEARS = {2:2010, 3:2011, 4:2014, 5:2017, 6:2020, 7:2023}

# ── 1. 표본 구성 변화 ──
print('\n[1] 표본 구성 비교 (W4 vs W5 vs W6)')
for w in [4, 5, 6]:
    s = df[df['wave']==w]
    print(f'  W{w} ({YEARS[w]}): N={len(s):,}')
    print(f'    Female: {s["female"].mean()*100:.1f}%')
    print(f'    Age: {s["age_raw"].mean():.1f} (SD={s["age_raw"].std():.1f})')
    print(f'    Wage worker: {s["wage"].mean()*100:.1f}%')
    print(f'    Long hrs: {s["long_hrs"].mean()*100:.1f}%')
    # 산업 분포
    top3 = s['industry'].value_counts().head(3)
    print(f'    Top industries: {dict(top3)}')
    # 인체공학 노출
    for erg in ['erg_posture_hi','erg_heavy_hi','erg_stand_hi','erg_repeat_hi']:
        print(f'    {erg}: {s[erg].mean()*100:.1f}%')
    print()

# ── 2. MSD 부위별 급감 패턴 ──
print('[2] MSD 부위별 유병률 변화')
for outcome, label in [('msd_any','Any MSD'),('msd_back_b','Back'),('msd_upper_b','Upper limb'),('msd_lower_b','Lower limb')]:
    rates = []
    for w in [4, 5, 6]:
        s = df[df['wave']==w]
        col = s[outcome].dropna()
        wts = s.loc[col.index, 'swt']
        r = np.average(col, weights=wts) * 100
        rates.append(f'W{w}={r:.1f}%')
    print(f'  {label:12s}: {", ".join(rates)}')

# ── 3. W5 표본 특성 차이 검정 ──
print(f'\n[3] W5 vs 나머지 웨이브 표본 특성 비교')
w5 = df[df['wave']==5]
others = df[df['wave']!=5]
for var, label in [('female','Female'), ('age_raw','Age'), ('long_hrs','Long hrs'),
                    ('wage','Wage worker'), ('erg_posture_hi','Erg posture hi')]:
    m5 = w5[var].mean()
    mo = others[var].mean()
    diff = m5 - mo
    print(f'  {label:18s}: W5={m5:.3f}  Others={mo:.3f}  diff={diff:+.3f}')

# ── 4. W5의 MSD 응답 분포 (원시값) ──
print(f'\n[4] MSD 원시값 분포 비교 (1=Yes, 2=No)')
for w in [4, 5, 6]:
    s = df[df['wave']==w]
    for v in ['msd_back','msd_upper','msd_lower']:
        vc = s[v].value_counts().sort_index()
        yes_pct = vc.get(1, 0) / (vc.get(1, 0) + vc.get(2, 0)) * 100
        print(f'  W{w} {v}: Yes={yes_pct:.1f}%  (1:{vc.get(1,0):,}, 2:{vc.get(2,0):,})')
    print()

# ── 5. 결론 ──
print('='*70)
print('W5 급감 가능 원인:')
print('  a) W5에서 N이 가장 큼 (30,752) → 표본 구성 변화 가능')
print('  b) 2017년 = 주 52시간 상한제 직전, 장시간 근로 비율 확인 필요')
print('  c) 설문 문항 순서/방식 변경 가능성 (KOSHA 확인 필요)')
print('  d) Sensitivity S5에서 W5 제외 시 결과 안정성 확인됨')
print('='*70)

# 저장
w5_analysis = {
    'Wave': [4, 5, 6],
    'Year': [2014, 2017, 2020],
    'N': [(df['wave']==w).sum() for w in [4,5,6]],
    'Any_MSD_pct': [df[df['wave']==w]['msd_any'].mean()*100 for w in [4,5,6]],
    'Female_pct': [df[df['wave']==w]['female'].mean()*100 for w in [4,5,6]],
    'Age_mean': [df[df['wave']==w]['age_raw'].mean() for w in [4,5,6]],
    'Long_hrs_pct': [df[df['wave']==w]['long_hrs'].mean()*100 for w in [4,5,6]],
}
pd.DataFrame(w5_analysis).to_csv(os.path.join(OUT_DIR, 'table_w5_analysis.csv'), index=False)
print(f'\n저장: table_w5_analysis.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7b — (D) Interaction Analysis: Autonomy x Ergonomic
# ══════════════════════════════════════════════════════════════
#
# Autonomy paradox의 핵심 질문:
#   "자율성이 낮은 근로자(사무직)의 MSD가 낮은 이유는
#    인체공학적 노출이 적기 때문인가?"
#
# 검증 방법:
#   1. Autonomy x Erg 교차항 → 효과 수정(effect modification) 검정
#   2. RERI (Relative Excess Risk due to Interaction) → 가산적 상호작용
#   3. 산업별 층화 분석 → 같은 산업 내에서 autonomy 효과 확인
#
# 참고: Lee et al. 2021, J Occup Health (KWCS 5차 interaction 분석)
# ══════════════════════════════════════════════════════════════

print('='*70)
print('(D) Interaction Analysis: Autonomy Paradox 설명')
print('='*70)

# ── 1. Autonomy x Erg_any 교차항 (Modified Poisson) ──
print('\n[1] Autonomy x Ergonomic Interaction (Any MSD)')

dfa['erg_any_hi'] = ((dfa['erg_posture_hi']==1) | (dfa['erg_heavy_hi']==1) |
                      (dfa['erg_stand_hi']==1) | (dfa['erg_repeat_hi']==1)).astype(float)
dfa['auto_x_erg'] = dfa['autonomy'] * dfa['erg_any_hi']

f_int = (f'msd_any ~ autonomy * erg_any_hi + psy_demand + support + '
         f'{covars} + wave_c + C(emp_type) + C(industry)')
m_int = smf.glm(f_int, data=dfa,
                  family=sm.families.Poisson(link=sm.families.links.Log()),
                  freq_weights=dfa['swt']).fit(cov_type='HC1')

print(f'  Autonomy main:       PR={np.exp(m_int.params["autonomy"]):.3f}  p={m_int.pvalues["autonomy"]:.4f}')
print(f'  Erg_any main:        PR={np.exp(m_int.params["erg_any_hi"]):.3f}  p={m_int.pvalues["erg_any_hi"]:.4f}')
print(f'  Autonomy x Erg:      PR={np.exp(m_int.params["autonomy:erg_any_hi"]):.3f}  p={m_int.pvalues["autonomy:erg_any_hi"]:.4f}')
print(f'  → p<0.05이면: 인체공학 노출 수준에 따라 autonomy의 효과가 달라짐')

# ── 2. 4분면 분석 (Karasek quadrant style) ──
print('\n[2] Autonomy x Ergonomic 4분면 분석')
print('    (autonomy: 1=자율있음, 2=없음; erg_any_hi: 0=저노출, 1=고노출)')

dfa['lo_auto_binary'] = (dfa['lo_auto']==1).astype(float)
quadrants = [
    ('Hi auto + Lo erg',  0, 0),  # Reference
    ('Hi auto + Hi erg',  0, 1),
    ('Lo auto + Lo erg',  1, 0),
    ('Lo auto + Hi erg',  1, 1),
]

# 유병률
print(f'\n  {"Quadrant":<25s} {"N":>8s} {"MSD%":>8s}')
print(f'  {"-"*45}')
for label, auto_val, erg_val in quadrants:
    sub = dfa[(dfa['lo_auto_binary']==auto_val) & (dfa['erg_any_hi']==erg_val)]
    rate = sub['msd_any'].mean()*100
    print(f'  {label:<25s} {len(sub):>8,} {rate:>7.1f}%')

# Modified Poisson for quadrants
dfa['q_hi_auto_hi_erg'] = ((dfa['lo_auto_binary']==0) & (dfa['erg_any_hi']==1)).astype(float)
dfa['q_lo_auto_lo_erg'] = ((dfa['lo_auto_binary']==1) & (dfa['erg_any_hi']==0)).astype(float)
dfa['q_lo_auto_hi_erg'] = ((dfa['lo_auto_binary']==1) & (dfa['erg_any_hi']==1)).astype(float)

f_quad = (f'msd_any ~ q_hi_auto_hi_erg + q_lo_auto_lo_erg + q_lo_auto_hi_erg + '
          f'psy_demand + support + {covars} + wave_c + C(emp_type) + C(industry)')
m_quad = smf.glm(f_quad, data=dfa,
                   family=sm.families.Poisson(link=sm.families.links.Log()),
                   freq_weights=dfa['swt']).fit(cov_type='HC1')

print(f'\n  Adjusted PR (ref: Hi autonomy + Lo erg exposure):')
for var, label in [('q_hi_auto_hi_erg','Hi auto + Hi erg'),
                    ('q_lo_auto_lo_erg','Lo auto + Lo erg'),
                    ('q_lo_auto_hi_erg','Lo auto + Hi erg')]:
    pr = np.exp(m_quad.params[var])
    ci = np.exp(m_quad.conf_int().loc[var])
    p = m_quad.pvalues[var]
    print(f'  {label:<25s} PR={pr:.2f} ({ci.iloc[0]:.2f}-{ci.iloc[1]:.2f}) p={p:.4f}')

# RERI 계산
pr_10 = np.exp(m_quad.params['q_hi_auto_hi_erg'])  # auto=0, erg=1
pr_01 = np.exp(m_quad.params['q_lo_auto_lo_erg'])   # auto=1, erg=0
pr_11 = np.exp(m_quad.params['q_lo_auto_hi_erg'])   # auto=1, erg=1
reri = pr_11 - pr_10 - pr_01 + 1
print(f'\n  RERI = {reri:.3f}')
print(f'  → RERI>0: 양의 가산적 상호작용 (두 위험의 결합효과가 개별 합보다 큼)')
print(f'  → RERI<0: 음의 가산적 상호작용')
print(f'  → RERI≈0: 상호작용 없음')

# ── 3. 산업별 층화: autonomy 효과 ──
print('\n[3] 산업별 Autonomy → MSD 효과 (top 5 산업)')

top_ind = dfa['industry'].value_counts().head(5).index.tolist()
ind_results = []
for ind in top_ind:
    sub = dfa[dfa['industry']==ind].copy()
    if len(sub) < 500: continue
    try:
        f_ind = f'msd_any ~ autonomy + psy_demand + support + {covars} + {erg_covars} + wave_c'
        m_ind = smf.glm(f_ind, data=sub,
                         family=sm.families.Poisson(link=sm.families.links.Log()),
                         freq_weights=sub['swt']).fit(cov_type='HC1')
        pr = np.exp(m_ind.params['autonomy'])
        ci = np.exp(m_ind.conf_int().loc['autonomy'])
        msd_rate = sub['msd_any'].mean()*100
        ind_results.append({
            'Industry': int(ind), 'N': len(sub), 'MSD%': round(msd_rate,1),
            'Autonomy_PR': round(pr,2), 'CI_lo': round(ci.iloc[0],2), 'CI_hi': round(ci.iloc[1],2),
            'p': round(m_ind.pvalues['autonomy'],4)
        })
    except:
        pass

df_ind = pd.DataFrame(ind_results)
print(df_ind.to_string(index=False))
df_ind.to_csv(os.path.join(OUT_DIR, 'table_interaction_industry.csv'), index=False)

# ── 4. 시각화: 4분면 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 4a. Quadrant bar chart
ax = axes[0]
labels = ['Hi auto\nLo erg\n(Ref)', 'Hi auto\nHi erg', 'Lo auto\nLo erg', 'Lo auto\nHi erg']
rates = []
for _, auto_val, erg_val in quadrants:
    sub = dfa[(dfa['lo_auto_binary']==auto_val) & (dfa['erg_any_hi']==erg_val)]
    rates.append(sub['msd_any'].mean()*100)
colors_q = ['#27ae60', '#e67e22', '#3498db', '#e74c3c']
bars = ax.bar(labels, rates, color=colors_q, alpha=0.7, edgecolor='black', linewidth=0.5)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('MSD Prevalence (%)', fontsize=11)
ax.set_title('(A) MSD by Autonomy x Ergonomic Exposure', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(rates)*1.15)
ax.grid(axis='y', alpha=0.3)

# 4b. Industry-stratified autonomy PR
ax = axes[1]
if len(df_ind) > 0:
    df_ind_sorted = df_ind.sort_values('Autonomy_PR')
    y_pos = range(len(df_ind_sorted))
    for i, (_, row) in enumerate(df_ind_sorted.iterrows()):
        color = '#e74c3c' if row['Autonomy_PR'] > 1 else '#3498db'
        ax.plot(row['Autonomy_PR'], i, 'o', color=color, markersize=10, zorder=3)
        ax.plot([row['CI_lo'], row['CI_hi']], [i, i], '-', color=color, linewidth=2.5, zorder=2)
        ax.annotate(f"Ind {int(row['Industry'])} (N={row['N']:,})",
                    (row['CI_hi']+0.01, i), fontsize=9, va='center')
    ax.axvline(x=1, color='gray', linestyle='--', alpha=0.7)
    ax.set_yticks([])
    ax.set_xlabel('Prevalence Ratio for Autonomy (per unit increase)', fontsize=10)
    ax.set_title('(B) Autonomy → MSD by Industry', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_interaction.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f'\n저장: fig_interaction.png, table_interaction_industry.csv')
print('='*70)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — Descriptive Table 1 (저널 제출용)
# ══════════════════════════════════════════════════════════════

def pct(col):
    s = col.dropna().sum()
    n = col.dropna().count()
    return f'{int(s):,} ({s/n*100:.1f})'

rows = []
for w in sorted(df['wave'].unique()):
    s = df[df['wave']==w]
    rows.append({
        'Wave': f'W{int(w)} ({YEARS[int(w)]})',
        'N': f'{len(s):,}',
        'Female': pct(s['female']),
        'Age': f'{s["age_raw"].mean():.1f} ({s["age_raw"].std():.1f})',
        'Any MSD': pct(s['msd_any']),
        'Hi demand': pct(s['psy_demand_hi']),
        'Lo autonomy': pct(s['lo_auto']),
        'Job strain': pct(s['job_strain']),
        'Long hrs': pct(s['long_hrs']),
    })

# Total
rows.append({
    'Wave': 'Total',
    'N': f'{len(df):,}',
    'Female': pct(df['female']),
    'Age': f'{df["age_raw"].mean():.1f} ({df["age_raw"].std():.1f})',
    'Any MSD': pct(df['msd_any']),
    'Hi demand': pct(df['psy_demand_hi']),
    'Lo autonomy': pct(df['lo_auto']),
    'Job strain': pct(df['job_strain']),
    'Long hrs': pct(df['long_hrs']),
})

df_t1 = pd.DataFrame(rows)
df_t1.to_csv(os.path.join(OUT_DIR, 'table1_descriptive.csv'), index=False)
print('Table 1: Descriptive Statistics by Wave')
print(df_t1.to_string(index=False))
print(f'\n저장: table1_descriptive.csv')
print('\n분석 완료!')